In [ ]:
%pip install -U  transformers datasets peft accelerate huggingface_hub safetensors  torch torchvision torchaudio torchao

In [5]:
# %pip install -U accelerate 
%pip install -U datasets


Note: you may need to restart the kernel to use updated packages.


# Mount drive

In [ ]:
IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    



# Import lib

In [9]:
import os
import json, torch
import pandas as pd
import json
import os
from pathlib import Path
from dotenv import load_dotenv

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model

 

In [10]:


BASE_MODEL = "ntphuc149/ViLegalQwen2.5-1.5B-Base"
OUTPUT_DIR = "./legal-lora"
HF_USERNAME = "tmtdev"
HF_REPO = f"{HF_USERNAME}/vn-legal-model"
HF_DATASET_REPO = f"{HF_USERNAME}/vn-legal-extraction"

MAX_LENGTH = 1024
DATA_DIR = "/content/drive/MyDrive/legal-ai/data"
DATA_PATH = os.path.join(DATA_DIR, "all_extract_citation_data.json")


# Directory Path

# Config class

In [ ]:
def load_json_file(file_path: str) -> list[dict]:
        """
        Load JSON data from a file.
        """
        if not Path(file_path).exists():
            print(f"File not found: {file_path}")
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data

legal_data = load_json_file(DATA_PATH)
print(len(legal_data))

def eda_data(data: list[dict]):
  df = pd.DataFrame(data=data)

  display(df.head())
  display(df.info())
  display(df.describe())

eda_data(legal_data)

In [ ]:
print("Cấu trúc 1 dòng mẫu:", legal_data[0])

require_columns = ["query_text", "question_text", "target"]

data = Dataset.from_list(legal_data).filter(
    lambda x: all(x.get(k) for k in ["query_text", "question_text", "target"])
)
print("Số dòng sau khi lọc:", len(data))

splits = data.train_test_split(test_size=0.2, seed=42)
temp = splits["test"].train_test_split(test_size=0.5, seed=42)

train_data = splits["train"]
val_data = temp["train"]
test_data = temp["test"]

print(len(train_data), len(val_data), len(test_data))

In [ ]:
# train example
print(train_data)

In [ ]:
from huggingface_hub import login, create_repo
from huggingface_hub.utils import validate_repo_id
from datasets import DatasetDict

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import notebook_login
notebook_login()

login()


dataset = DatasetDict({
    "train": train_data,
    "validation": val_data,
    "test": test_data
})

# Push
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_DATASET_REPO = "tmtdev/vn-legal-extraction"
print(HF_TOKEN)
dataset.push_to_hub(
    repo_id=HF_DATASET_REPO,
    token=HF_TOKEN,
    private=False
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = """
Bạn là trợ lý xử lý truy vấn pháp luật Việt Nam.
Dựa vào dữ liệu đã học, hãy tạo JSON hợp lệ gồm:
query_text, question_text, context.
Không thêm nội dung ngoài JSON.
"""

In [ ]:
def preprocess(example):
    target = json.dumps({
        "query_text": example["query_text"],
        "question_text": example["question_text"],
        "target": example["target"]
    }, ensure_ascii=False)

    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["query_text"]}
    ]
    full = prompt + [{"role": "assistant", "content": target}]

    prompt_ids = tokenizer.apply_chat_template(
        prompt, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(
        full, tokenize=True, add_generation_prompt=False
    )

    full_ids = full_ids[:MAX_LENGTH]
    labels = [-100] * min(len(prompt_ids), len(full_ids))
    labels += full_ids[len(labels):]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }

train_data = train_data.map(preprocess, remove_columns=train_data.column_names)
val_data = val_data.map(preprocess, remove_columns=val_data.column_names)
test_data = test_data.map(preprocess, remove_columns=test_data.column_names)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM"
))

model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        label_pad_token_id=-100
    )
)

trainer.train()

In [ ]:
print("Validation:", trainer.evaluate(val_data))
print("Test:", trainer.evaluate(test_data))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
from huggingface_hub import login
login()

model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)

In [ ]:
def predict(query_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        return result

predict("Theo khoản 2 Điều 13 Nghị định 10/2020/NĐ-CP, thời gian lưu trữ hình ảnh là bao lâu?")

In [ ]:
# !pip install datasets

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM
import accelerate

print(accelerate.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"

# MODEL_ID = "ntphuc149/ViLegalQwen2.5-1.5B-Base"
HF_USERNAME = "tmtdev"
MODEL_ID = f"{HF_USERNAME}/vn-legal-model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

SYSTEM_PROMPT = """
Bạn là hệ thống trích xuất cấu trúc truy vấn pháp luật Việt Nam.

Đầu vào: query_text của người dùng.
Đầu ra: duy nhất một JSON hợp lệ theo cấu trúc:

{
  "query_text": "string",
  "question_text": "string",
  "target": {
    "part": null,
    "chapter": null,
    "section": null,
    "subsection": null,
    "article": null,
    "clause": null,
    "point": null,
    "document": null
  }
}

Quy tắc:
- query_text: giữ nguyên chính xác câu đầu vào.
- question_text: giữ nội dung câu hỏi, loại bỏ phần trích dẫn pháp luật
  như tên văn bản, Phần, Chương, Mục, Tiểu mục, Điều, Khoản, Điểm.
  Không làm thay đổi ý nghĩa câu hỏi.
- target: trích xuất chính xác các cấp trích dẫn xuất hiện trong query_text.
- document: tên văn bản pháp luật được nhắc đến trong query_text.
- Các cấp không được nhắc đến phải là null.
- Giá trị cấp trích dẫn là chuỗi, ví dụ "I", "13", "2", "a".
- Không suy đoán cấp trích dẫn còn thiếu.
- Không tự tra cứu hoặc tự bổ sung thông tin pháp luật.
- Chỉ trả về JSON, không thêm Markdown hay giải thích.
"""


SYSTEM_PROMPT = """
Bạn là trợ lý xử lý truy vấn pháp luật Việt Nam.
Dựa vào dữ liệu đã học, hãy tạo JSON hợp lệ gồm:
query_text, question_text, context.
Không thêm nội dung ngoài JSON.
"""

def extract_legal_structure(query_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        # return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Use inputs["input_ids"] to slice the output tokens correctly
    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print("RAW OUTPUT:")
    print(result)

    try:
        parsed = json.loads(result)
        print("\nPARSED JSON:")
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
        return parsed
    except json.JSONDecodeError:
        print("\nModel chưa trả về JSON hợp lệ.")
        return None

extract_legal_structure(
    "Chương I của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam "
    "năm 2013 quy định về vấn đề gì?"
)


In [25]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Tổng số tham số: {total_params:,}") 
# Kết quả sẽ ra khoảng ~1,540,000,000 (tức là 1.54B)

# Hoặc in thông tin chi tiết cấu hình mô hình
print(model.config)

Tổng số tham số: 1,562,179,072
Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 131072,
  "max_window_layers": 

In [ ]:

import json
import torch


SYSTEM_PROMPT = """
Bạn là trợ lý xử lý truy vấn pháp luật Việt Nam.
Dựa vào dữ liệu đã học, hãy tạo JSON hợp lệ gồm:
query_text, question_text, context.
Không thêm nội dung ngoài JSON.
"""

def predict(query_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Qwen dùng <|im_end|> để kết thúc một lượt hội thoại
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    stop_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )

    input_length = inputs["input_ids"].shape[-1]

    result = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    # Nếu model sinh thêm nội dung sau JSON, chỉ lấy object JSON đầu tiên
    try:
        decoder = json.JSONDecoder()
        
        start = result.find("{")

        if start == -1:
            return {"error": "Model không sinh JSON hợp lệ", "raw_output": result}

        parsed, _ = decoder.raw_decode(result[start:])
        return parsed

    except json.JSONDecodeError:
        return {"error": "Không thể parse JSON", "raw_output": result}


result = predict(
    "Theo khoản 2 Điều 13 Nghị định 10/2020/NĐ-CP, thời gian lưu trữ hình ảnh là bao lâu?"
)

print(json.dumps(result, ensure_ascii=False, indent=2))

In [20]:
from datasets import load_dataset

raw_test = load_dataset(HF_DATASET_REPO, split="test")

print("Số lượng mẫu test:", len(raw_test))
print("Ví dụ mẫu test:", raw_test[0])

Số lượng mẫu test: 1055
Ví dụ mẫu test: {'query_text': 'Nội dung Khoản 1 Điều 157 theo Bộ luật Dân sự 2015 là gì?', 'question_text': 'Nội dung Khoản 1 Điều 157 theo  là gì?', 'target': {'part': None, 'chapter': None, 'section': None, 'subsection': None, 'article': '157', 'clause': '1', 'point': None, 'document': 'Bộ luật Dân sự 2015'}}


In [24]:
# SYSTEM_PROMPT phải giống lúc train
# Ví dụ dưới đây chỉ dùng nếu đây chính là prompt đã train.
# SYSTEM_PROMPT = "..."

import re

def parse_json_output(text):
    """Lấy object JSON đầu tiên trong output của model."""
    text = text.strip()

    # Loại bỏ Markdown code fence nếu có
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    decoder = json.JSONDecoder()
    start = text.find("{")

    if start == -1:
        return None

    try:
        obj, _ = decoder.raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None


def predict(query_text, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Chỉ lấy phần model sinh mới, không lấy lại prompt
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return output_text, parse_json_output(output_text)
def predict(query_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Qwen dùng <|im_end|> để kết thúc một lượt hội thoại
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    stop_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )

    input_length = inputs["input_ids"].shape[-1]

    result = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    # Nếu model sinh thêm nội dung sau JSON, chỉ lấy object JSON đầu tiên
    try:
        decoder = json.JSONDecoder()
        
        start = result.find("{")

        if start == -1:
            print("Model không sinh JSON hợp lệ, trả về raw output.")
            return result, None

        parsed, _ = decoder.raw_decode(result[start:])
        return result, parsed

    except json.JSONDecodeError:
        print("Không thể parse JSON, trả về raw output.")
        return result, None



# Chạy đánh giá
results = []
test = raw_test.to_list()[:5]  # Giới hạn 100 mẫu để test nhanh
print(test[0])
for i, example in enumerate(test):
    if i + 1 % 10 == 0:
        print(f"Đang đánh giá mẫu {i+1}/{len(raw_test)}...")
    print(f"Query: {example['query_text']}")
    query = example["query_text"]
    output_text, pred = predict(query)

    expected_target = example["target"]
    expected_question = example["question_text"]

    valid_json = pred is not None
    correct_keys = (
        valid_json
        and set(pred.keys()) == {
            "query_text", "question_text", "target"
        }
    )

    query_ok = (
        correct_keys
        and pred["query_text"] == query
    )

    question_ok = (
        correct_keys
        and isinstance(pred["question_text"], str)
        and bool(pred["question_text"].strip())
    )

    target_ok = (
        correct_keys
        and isinstance(pred["target"], type(expected_target))
        and (
            bool(pred["target"].strip())
            if isinstance(pred["target"], str)
            else pred["target"] is not None
        )
    )

    results.append({
        "query_text": query,
        "expected_question": expected_question,
        "predicted_question": (
            pred.get("question_text") if valid_json else None
        ),
        "expected_target": expected_target,
        "predicted_target": (
            pred.get("target") if valid_json else None
        ),
        "valid_json": valid_json,
        "correct_keys": correct_keys,
        "query_exact_match": query_ok,
        "question_nonempty": question_ok,
        "question_exact_match": (
            pred.get("question_text") == expected_question
            if correct_keys else False
        ),
        "target_format_ok": target_ok,
        "target_exact_match": (
            pred.get("target") == expected_target
            if correct_keys else False
        ),
        "all_checks_pass": (
            query_ok and question_ok and target_ok
        ),
        "raw_output": output_text
    })

df = pd.DataFrame(results)

# Tổng hợp các metric
metrics = [
    "valid_json",
    "correct_keys",
    "query_exact_match",
    "question_nonempty",
    "question_exact_match",
    "target_format_ok",
    "target_exact_match",
    "all_checks_pass"
]

summary = pd.DataFrame({
    "metric": metrics,
    "score_percent": [
        100 * df[m].mean() for m in metrics
    ]
})

print(summary.to_string(index=False))

# Xem những mẫu model dự đoán sai
errors = df[~df["all_checks_pass"]]

display(errors.head(10))

# Lưu kết quả để phân tích
df.to_csv(
    "test_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)

{'query_text': 'Nội dung Khoản 1 Điều 157 theo Bộ luật Dân sự 2015 là gì?', 'question_text': 'Nội dung Khoản 1 Điều 157 theo  là gì?', 'target': {'part': None, 'chapter': None, 'section': None, 'subsection': None, 'article': '157', 'clause': '1', 'point': None, 'document': 'Bộ luật Dân sự 2015'}}
Query: Nội dung Khoản 1 Điều 157 theo Bộ luật Dân sự 2015 là gì?
Query: Nội dung Điều 80 theo Bộ luật Dân sự 2015 là gì?
Query: Khoản 4 Điều 218, thuộc Chương XIII Phần thứ hai, theo Bộ luật Dân sự 2015 quy định gì?
Query: Khoản 3 Điều 599 theo Bộ luật Dân sự 2015 quy định gì?
Query: Theo Điều 34 Quy định 69-QĐ/TW năm 2022 , Nếu đảng viên không báo cáo về việc kết hôn của con mình với người nước ngoài thì bị xử lý kỷ luật thế nào?
              metric  score_percent
          valid_json          100.0
        correct_keys          100.0
   query_exact_match          100.0
   question_nonempty          100.0
question_exact_match          100.0
    target_format_ok          100.0
  target_exact_

,query_text,expected_question,predicted_question,expected_target,predicted_target,valid_json,correct_keys,query_exact_match,question_nonempty,question_exact_match,target_format_ok,target_exact_match,all_checks_pass,raw_output
